# How Many People in the World Can Squat 315 / 405 / 495?
### A data-driven estimate — 3+ million real competition lifts, 8.2 billion people

> **315 lbs. Three plates a side.**  
> In the gym people talk about it the same way they talk about benching 225.  
> But squat numbers are the ones that *really* separate gym-goers.  
> And when you zoom out to 8.2 billion people — including every kid, grandparent, and person who's never touched a barbell — the number gets uncomfortable fast.

---
**Method:**  
OpenPowerlifting data (3M+ records) → fit squat strength distribution → correct for competitor vs. casual-gym gap → account for the fraction who actually back squat → 10-region world model → 100,000 Monte Carlo trials

---
## ⚙️ Setup
*Theme, imports, helpers.*

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import lognorm as _lognorm
import warnings, requests, zipfile, io, os
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Dark gym aesthetic ──────────────────────────────────────────────────────
BG    = '#0d0d0d'
PANEL = '#141414'
GRID  = '#252525'
WHITE = '#f0f0f0'
GRAY  = '#666666'
C315  = '#00b4ff'   # blue  — entry-level milestone
C405  = '#ff2d2d'   # red   — serious strength
C495  = '#ffd700'   # gold  — elite
MALE  = '#4fc3f7'
FEM   = '#f48fb1'

MILESTONES = [315, 405, 495]
M_COLORS   = [C315, C405, C495]
M_TAGS     = ['315 lbs', '405 lbs', '495 lbs']
M_LABELS   = ['315 lbs\n(3 plates)', '405 lbs\n(4 plates)', '495 lbs\n(5 plates)']
FRAC_MALE  = 0.504
WORLD_POP  = 8_200_000_000

# ── IG Reel canvas: 9:16 portrait ──────────────────────────────────────────
REEL_W, REEL_H = 6.75, 12.0   # inches at 160 dpi → 1080×1920
REEL_DPI = 160

def make_fig(w=13, h=6, cols=1):
    if cols == 1:
        fig, ax = plt.subplots(figsize=(w, h), facecolor=BG)
        _style(ax); return fig, ax
    fig, axes = plt.subplots(1, cols, figsize=(w, h), facecolor=BG)
    [_style(ax) for ax in axes]; return fig, axes

def make_reel(title_lines=None):
    """Portrait 9:16 figure for IG Reels."""
    fig, ax = plt.subplots(figsize=(REEL_W, REEL_H), facecolor=BG)
    _style(ax)
    if title_lines:
        y = 0.97
        for line, fs, fw in title_lines:
            fig.text(0.5, y, line, ha='center', va='top',
                     color=WHITE, fontsize=fs, fontweight=fw,
                     transform=fig.transFigure)
            y -= 0.045
    return fig, ax

def _style(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=GRAY, labelsize=11)
    for sp in ax.spines.values(): sp.set_color(GRID)
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)
    ax.title.set_color(WHITE)
    ax.grid(color=GRID, lw=0.6, alpha=0.8)

def p_above(target, sigma_arr, median_arr):
    z = (np.log(target) - np.log(median_arr)) / sigma_arr
    return stats.norm.sf(z)

print('Ready.')

---
## 📊 Chapter 1 — The Data

Same OpenPowerlifting database as the bench model — 3.9M+ records.  
This time we pull `Best3SquatKg`, raw/wraps only, same quality filters.

In [ ]:
OPL_PATH = 'opl_data.csv'

if not os.path.exists(OPL_PATH):
    print('Downloading OpenPowerlifting dataset (~200MB, one-time)...')
    url = 'https://openpowerlifting.gitlab.io/opl-csv/files/openpowerlifting-latest.zip'
    r   = requests.get(url, stream=True, timeout=180)
    total, chunks, downloaded = int(r.headers.get('content-length', 0)), [], 0
    for chunk in r.iter_content(chunk_size=1024*256):
        chunks.append(chunk); downloaded += len(chunk)
        if total: print(f'  {downloaded/1e6:.0f} / {total/1e6:.0f} MB', end='\r')
    with zipfile.ZipFile(io.BytesIO(b''.join(chunks))) as z:
        csv_name = [n for n in z.namelist() if n.endswith('.csv')][0]
        with z.open(csv_name) as f:
            df_raw = pd.read_csv(f, low_memory=False)
    df_raw.to_csv(OPL_PATH, index=False)
    print(f'\nSaved → {OPL_PATH}')
else:
    df_raw = pd.read_csv(OPL_PATH, low_memory=False)
    print(f'Loaded from cache — {len(df_raw):,} records')

In [ ]:
# Squat-specific filter: Best3SquatKg, raw/wraps only, valid lifts
df = df_raw[['Sex','Best3SquatKg','Equipment']].copy()
df = df[df['Best3SquatKg'] > 0].dropna()
df['squat_lbs'] = df['Best3SquatKg'] * 2.20462
df = df[df['Equipment'].isin(['Raw','Wraps'])]
df = df[df['Sex'].isin(['M','F'])]
# Reasonable squat range: 45 lbs (empty bar attempt) to 900 lbs
df = df[(df['squat_lbs'] >= 45) & (df['squat_lbs'] <= 900)]

male_opl   = df[df['Sex']=='M']['squat_lbs'].values
female_opl = df[df['Sex']=='F']['squat_lbs'].values

print(f'Clean squat records:  Male {len(male_opl):,}  |  Female {len(female_opl):,}')
print(f'Male   — median {np.median(male_opl):.1f} lbs  |  mean {np.mean(male_opl):.1f} lbs')
print(f'Female — median {np.median(female_opl):.1f} lbs  |  mean {np.mean(female_opl):.1f} lbs')

---
## 📐 Chapter 2 — The Shape of Squat Strength

Same log-normal model: strength can't go negative, the top end has no hard ceiling, and the log of squat weights follows a bell curve.  

One key difference vs. bench: squat strength is meaningfully higher in absolute terms (most trained males squat more than they bench), so the milestones 315/405/495 cut deeper into the tail than 225/315/405 for bench.

In [ ]:
m_shape, _, m_scale = _lognorm.fit(male_opl,   floc=0)
f_shape, _, f_scale = _lognorm.fit(female_opl, floc=0)

print(f'Male   sigma={m_shape:.4f}  fit_median={m_scale:.1f} lbs  data_median={np.median(male_opl):.1f}')
print(f'Female sigma={f_shape:.4f}  fit_median={f_scale:.1f} lbs  data_median={np.median(female_opl):.1f}')

x = np.linspace(45, 800, 600)
fig, axes = make_fig(13, 5, cols=2)
for ax, vals, sh, sc, color, label in [
    (axes[0], male_opl,   m_shape, m_scale, MALE, 'Male'),
    (axes[1], female_opl, f_shape, f_scale, FEM,  'Female'),
]:
    ax.hist(vals, bins=150, density=True, color=color, alpha=0.35, edgecolor='none')
    ax.plot(x, _lognorm.pdf(x, sh, scale=sc), color=WHITE, lw=2.2, label='Log-normal fit')
    ax.axvline(np.median(vals), color=color, lw=1.4, ls='--', alpha=0.8,
               label=f'Median {np.median(vals):.0f} lbs')
    ax.set_title(f'{label} — Raw OPL Squat', color=WHITE, fontsize=12, fontweight='bold')
    ax.set_xlabel('Squat (lbs)'); ax.set_ylabel('Density')
    ax.set_xlim(45, 800)
    ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
    ax.text(0.97, 0.92,
            f'σ = {sh:.3f}\nmedian = {np.median(vals):.0f} lbs\nn = {len(vals):,}',
            transform=ax.transAxes, ha='right', va='top', color=WHITE, fontsize=10,
            bbox=dict(facecolor=GRID, edgecolor='none', pad=5))

# Mark milestones
for ax in axes:
    for t, c in zip(MILESTONES, M_COLORS):
        ax.axvline(t, color=c, lw=1.2, ls=':', alpha=0.7)

fig.suptitle('Log-Normal Fit — OpenPowerlifting Raw Squat Data',
             color=WHITE, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('01_opl_fit.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🎯 Chapter 3 — Two Gaps Nobody Talks About

**Gap 1 — Competitors vs. gym-goers:**  
OPL athletes are self-selected peak performers. A casual gym-goer squats much less.  
We keep the OPL-fitted sigma (shape) but shift the median down to normative gym-goer data.

**Gap 2 — The squat fraction:**  
Not everyone in the gym does barbell back squats. Many use Smith machines, leg press, or avoid legs entirely.  
Male back squat fraction among resistance trainers: ~50-65% (lower than bench press at 60-78%).  
Female back squat fraction: ~30-42% (higher relative to bench because squat rack culture is stronger for women's lower-body work).

**Gap 3 — The denominator:**  
The world has 8.2 billion people. Most are children, elderly, or live in regions where gyms barely exist.  
We model all of them — because when the question is 'how many people on Earth can squat 405,' the denominator is everyone.

In [ ]:
fig, ax = make_fig(12, 5)
x = np.linspace(20, 800, 600)

# Gym-goer medians for squat (NSCA normative + WHO BW scaling for North America baseline)
# Male casual gym-goer squat median: ~205 lbs (vs OPL competitor ~360+)
# Female casual gym-goer squat median: ~115 lbs
gym_m_med, gym_f_med = 205.0, 115.0
gym_sig_m, gym_sig_f = 0.33, 0.345

for med, sigma, color, label, ls in [
    (np.median(male_opl),   m_shape,   MALE, 'Male — OPL competitors',        '-'),
    (gym_m_med,             gym_sig_m, MALE, 'Male — gym-goer (N. America)',  '--'),
    (np.median(female_opl), f_shape,   FEM,  'Female — OPL competitors',      '-'),
    (gym_f_med,             gym_sig_f, FEM,  'Female — gym-goer (N. America)','--'),
]:
    ax.plot(x, _lognorm.pdf(x, sigma, scale=med),
            color=color, lw=2, ls=ls, label=label, alpha=0.9)

y_top = ax.get_ylim()[1]
for t, c in zip(MILESTONES, M_COLORS):
    ax.axvline(t, color=c, lw=1.2, ls=':', alpha=0.6)
    ax.text(t + 4, y_top * 0.03, f'{t}', color=c, fontsize=9)

ax.set_xlabel('Squat (lbs)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Competitors vs. Gym-Goers: Why the Median Shifts Down',
             color=WHITE, fontsize=13, fontweight='bold')
ax.legend(fontsize=9.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.set_xlim(20, 800)
plt.tight_layout()
plt.savefig('02_adjustment.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🌍 Chapter 4 — 10-Region World Model

Same regional structure as the bench model, recalibrated for squat:
- **Lift rates** (resistance training participation): same as bench model
- **Squat fractions**: lower than bench for males, relatively higher for females
- **Squat medians**: scaled up from bench medians (~squat ≈ 1.25–1.35× bench at gym-goer level)
- **Denominator**: full world population 16-70 (4.674B) — this is the honest denominator

In [ ]:
# Regional parameters — squat edition
#
# MSquatFrac / FSquatFrac: fraction of resistance trainers who do barbell back squat.
#   Males ~50-65%: bench press is still more popular than back squat in most gyms;
#     many males use Smith machine, hack squat, or skip legs.
#   Females ~30-42%: squat rack has become more culturally prominent for women (glute
#     training), though form and access still limit barbell squat adoption.
#
# Squat medians: NSCA + WHO regional BW scaling. Squat is ~1.25-1.35x bench at
#   gym-goer level for males; ~1.35-1.45x for females (females' squat:bench ratio
#   tends to be higher because lower-body strength advantage is larger).
#
# Columns: name, pop_16_70 (M), lift_mode, lift_lo, lift_hi,
#          msf, fsf (squat fracs),
#          male_med, male_lo, male_hi, female_med, female_lo, female_hi

REGIONS = [
    #  name                     pop   lm     ll     lh   msf   fsf   mm   mlo  mhi   fm  flo  fhi
    ('North America',           210, 0.22, 0.17,  0.28, 0.62, 0.38, 205, 185, 225, 115,  98, 132),
    ('Europe',                  450, 0.15, 0.10,  0.20, 0.58, 0.34, 190, 172, 208, 108,  92, 124),
    ('Russia / C. Asia',        165, 0.10, 0.07,  0.14, 0.60, 0.34, 185, 167, 203, 105,  89, 121),
    ('East Asia',               950, 0.07, 0.04,  0.11, 0.50, 0.30, 160, 144, 176,  90,  76, 104),
    ('Latin America',           380, 0.08, 0.05,  0.12, 0.58, 0.36, 165, 149, 181,  93,  79, 107),
    ('Middle East / N. Af.',    310, 0.05, 0.03,  0.08, 0.54, 0.28, 165, 149, 181,  88,  74, 102),
    ('South Asia',             1080, 0.025,0.015, 0.040,0.50, 0.28, 140, 125, 155,  77,  65,  89),
    ('SE Asia',                 500, 0.035,0.020, 0.055,0.52, 0.30, 148, 133, 163,  83,  70,  96),
    ('Sub-Saharan Africa',      610, 0.015,0.008, 0.025,0.50, 0.28, 148, 133, 163,  83,  70,  96),
    ('Oceania',                  19, 0.20, 0.16,  0.25, 0.60, 0.36, 198, 178, 218, 112,  95, 129),
]

cols = ['Region','Pop_M','LiftMode','LiftLo','LiftHi','MSquatFrac','FSquatFrac',
        'MMed','MMedLo','MMedHi','FMed','FMedLo','FMedHi']
df_r = pd.DataFrame(REGIONS, columns=cols)
df_r['Pop_M'] *= 1e6

total_pop    = df_r['Pop_M'].sum()
world_1670   = WORLD_POP * 0.57
print(f'Regional pop sum:  {total_pop/1e9:.3f}B')
print(f'World 16-70 (UN):  {world_1670/1e9:.3f}B')
print(f'Coverage:          {total_pop/world_1670*100:.1f}%')

df_r['EffSF']      = FRAC_MALE * df_r['MSquatFrac'] + (1-FRAC_MALE) * df_r['FSquatFrac']
df_r['Squatters_M'] = df_r['Pop_M'] * df_r['LiftMode'] * df_r['EffSF'] / 1e6
print(f'\nTotal barbell-squatters (point estimate): {df_r["Squatters_M"].sum():.0f}M')
print(f'vs. world 16-70: {df_r["Squatters_M"].sum()*1e6/world_1670*100:.2f}%')
print(f'vs. full world:  {df_r["Squatters_M"].sum()*1e6/WORLD_POP*100:.3f}%')

In [ ]:
# ── Squatter pool by region + TOTAL ────────────────────────────────────────
total_sq = df_r['Squatters_M'].sum()
all_regions = df_r['Region'].tolist() + ['TOTAL']
all_vals    = df_r['Squatters_M'].tolist() + [total_sq]
bar_cols    = [C315] * len(df_r) + [C495]

fig, ax = make_fig(13, 5.5)
y = np.arange(len(all_regions))
ax.axhline(len(df_r) - 0.5, color=GRID, lw=1.2, ls='--', alpha=0.7)

for i, (val, c) in enumerate(zip(all_vals, bar_cols)):
    ax.barh(i, val, color=c, alpha=0.82, edgecolor='none', height=0.6)
    ax.text(val + 0.2, i, f'{val:.1f}M', va='center', color=WHITE, fontsize=10,
            fontweight='bold' if i == len(all_regions) - 1 else 'normal')

ax.set_yticks(y)
ax.set_yticklabels(all_regions, color=WHITE, fontsize=10)
ax.set_xlabel('Estimated barbell squatters (millions)', fontsize=11)
ax.set_title('Barbell Squat Pool by Region  (point estimate)',
             color=WHITE, fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('03_regions.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## 🎲 Chapter 5 — Monte Carlo: 100,000 Trials

Every input is uncertain. We draw each parameter from its plausible range 100,000 times and record the distribution of outcomes.

| Parameter | Distribution | Basis |
|---|---|---|
| Lift rate per region | Triangular(lo, mode, hi) | IHRSA, CDC, Eurobarometer |
| Male squat frac | Uniform(MSF ± 15%) | gym behavior data |
| Female squat frac | Uniform(FSF ± 15%) | gym behavior data |
| Male median 1RM | Uniform(lo, hi) | NSCA normative + BW scaling |
| Female median 1RM | Uniform(lo, hi) | NSCA normative + BW scaling |
| Spread σ (male) | Uniform(0.28, 0.38) | calibrated (OPL is upper bound; NSCA trained sets floor) |
| Spread σ (female) | Uniform(0.29, 0.40) | same calibration |

> Sigma rationale: same as bench model — OPL sigma inflates the tail via elite competitors.  
> NSCA trained male squat P10≈165, P50≈235, P90≈330 implies σ≈0.28. Casual populations are wider; ceiling set at 0.38.

In [ ]:
N_SIM = 100_000
R     = len(df_r)
rng   = np.random.default_rng(42)

pop   = df_r['Pop_M'].values
msf_v = df_r['MSquatFrac'].values
fsf_v = df_r['FSquatFrac'].values

lift_s  = rng.triangular(df_r['LiftLo'].values, df_r['LiftMode'].values,
                          df_r['LiftHi'].values, size=(N_SIM, R))
msf_s   = rng.uniform(msf_v * 0.85, msf_v * 1.15, size=(N_SIM, R))
fsf_s   = rng.uniform(fsf_v * 0.85, fsf_v * 1.15, size=(N_SIM, R))
mmed_s  = rng.uniform(df_r['MMedLo'].values, df_r['MMedHi'].values, size=(N_SIM, R))
fmed_s  = rng.uniform(df_r['FMedLo'].values, df_r['FMedHi'].values, size=(N_SIM, R))
m_sig_s = rng.uniform(0.28, 0.38, size=(N_SIM, 1))
f_sig_s = rng.uniform(0.29, 0.40, size=(N_SIM, 1))

male_cnt   = pop * lift_s * msf_s * FRAC_MALE
female_cnt = pop * lift_s * fsf_s * (1 - FRAC_MALE)

mc_results = {}
for target in MILESTONES:
    pm = stats.norm.sf((np.log(target) - np.log(mmed_s)) / m_sig_s)
    pf = stats.norm.sf((np.log(target) - np.log(fmed_s)) / f_sig_s)
    mc_results[target] = (male_cnt * pm + female_cnt * pf).sum(axis=1) / 1e6

mc_per_region = {}
for target in MILESTONES:
    pm = stats.norm.sf((np.log(target) - np.log(mmed_s)) / m_sig_s)
    pf = stats.norm.sf((np.log(target) - np.log(fmed_s)) / f_sig_s)
    mc_per_region[target] = (male_cnt * pm + female_cnt * pf) / 1e6

print(f'Simulation complete — {N_SIM:,} trials\n')
for t in MILESTONES:
    v   = mc_results[t]
    pct = np.median(v) / (WORLD_POP / 1e6) * 100
    print(f'  {t} lbs: {np.median(v):.2f}M  '
          f'[{np.percentile(v,2.5):.2f}M – {np.percentile(v,97.5):.2f}M]  '
          f'({pct:.4f}% of world)')

---
## 🎬 IG Reel Slides

Portrait 9:16 charts optimized for Instagram Reels.  
Large text, mobile-readable, dark aesthetic.  
**Slide 00** — Hook  
**Slides 01-03** — Milestone reveals  
**Slide 04** — Drop-off curve (the 'wow' frame)  
**Slide 05** — Comparison to bench numbers (context/retention)  
**Slide 06** — Regional breakdown (where in the world)  

In [ ]:
# ── Shared reveal helper ─────────────────────────────────────────────────────
_mw = df_r['Pop_M'] * df_r['LiftMode'] * df_r['MSquatFrac']
_fw = df_r['Pop_M'] * df_r['LiftMode'] * df_r['FSquatFrac']
_global_m_med = float(np.average(df_r['MMed'], weights=_mw))
_global_f_med = float(np.average(df_r['FMed'], weights=_fw))
_m_sig, _f_sig = 0.33, 0.345

def reveal_chart_reel(target, color, ax, plate_label):
    """Draw the distribution reveal optimized for portrait/reel layout."""
    sims   = mc_results[target]
    med    = np.median(sims)
    lo, hi = np.percentile(sims, [2.5, 97.5])
    pct    = med / (WORLD_POP / 1e6) * 100

    x = np.linspace(20, 750, 1200)
    pdf_m = _lognorm.pdf(x, _m_sig, scale=_global_m_med)
    pdf_f = _lognorm.pdf(x, _f_sig, scale=_global_f_med)

    ax.plot(x, pdf_m, color=MALE, lw=2.5, alpha=0.85, label='Male')
    ax.plot(x, pdf_f, color=FEM,  lw=2.5, alpha=0.85, label='Female')

    mask = x >= target
    ax.fill_between(x[mask], pdf_m[mask], color=MALE, alpha=0.30)
    ax.fill_between(x[mask], pdf_f[mask], color=FEM,  alpha=0.30)
    ax.axvline(target, color=color, lw=3.0, ls='--', alpha=0.95)

    y_top = max(pdf_m.max(), pdf_f.max())
    ax.text(target + 8, y_top * 0.90,
            f'{target} lbs\n{plate_label}', color=color,
            fontsize=14, fontweight='bold', va='top')

    # Big stat box in middle of chart
    ax.text(0.5, 0.52,
            f'≈ {med:.2f}M people',
            transform=ax.transAxes, ha='center', va='center',
            color=color, fontsize=22, fontweight='black')
    ax.text(0.5, 0.42,
            f'that\'s {pct:.4f}%\nof all humans',
            transform=ax.transAxes, ha='center', va='center',
            color=WHITE, fontsize=14, fontweight='bold')
    ax.text(0.5, 0.31,
            f'95% CI: {lo:.2f}M – {hi:.2f}M',
            transform=ax.transAxes, ha='center', va='center',
            color=GRAY, fontsize=11)

    ax.set_xlim(20, 750)
    ax.set_xlabel('Squat (lbs)', fontsize=12, labelpad=6)
    ax.legend(fontsize=11, facecolor=PANEL, labelcolor=WHITE,
              edgecolor=GRID, loc='upper right')

In [ ]:
# ── Slide 00 — HOOK ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0, 0, 1, 1])   # full canvas
ax.set_facecolor(BG)
ax.axis('off')

# Top accent line
ax.axhline(0.93, color=C315, lw=4, xmin=0.08, xmax=0.92)

# Hook question
ax.text(0.5, 0.85, 'How many people',
        ha='center', va='center', color=WHITE,
        fontsize=34, fontweight='black', transform=ax.transAxes)
ax.text(0.5, 0.77, 'on Earth can',
        ha='center', va='center', color=WHITE,
        fontsize=34, fontweight='black', transform=ax.transAxes)
ax.text(0.5, 0.68, 'SQUAT',
        ha='center', va='center', color=C315,
        fontsize=56, fontweight='black', transform=ax.transAxes)
ax.text(0.5, 0.59, '3 plates?',
        ha='center', va='center', color=WHITE,
        fontsize=44, fontweight='black', transform=ax.transAxes)

# Sub hook
ax.axhline(0.53, color=GRID, lw=1.5, xmin=0.15, xmax=0.85, alpha=0.6)
ax.text(0.5, 0.47, 'I modeled 8.2 billion people',
        ha='center', va='center', color=GRAY,
        fontsize=17, transform=ax.transAxes)
ax.text(0.5, 0.41, 'using 3M+ real competition lifts',
        ha='center', va='center', color=GRAY,
        fontsize=17, transform=ax.transAxes)
ax.text(0.5, 0.34, 'The answer will humble you.',
        ha='center', va='center', color=WHITE,
        fontsize=19, fontweight='bold', transform=ax.transAxes)

# Milestone pills
for i, (m, c, lbl) in enumerate(zip(MILESTONES, M_COLORS, ['3 plates','4 plates','5 plates'])):
    x_pos = 0.18 + i * 0.32
    ax.text(x_pos, 0.22, f'{m}', ha='center', va='center',
            color=c, fontsize=26, fontweight='black', transform=ax.transAxes)
    ax.text(x_pos, 0.16, lbl, ha='center', va='center',
            color=GRAY, fontsize=13, transform=ax.transAxes)

# Bottom CTA
ax.text(0.5, 0.07, 'Watch to the end for 495 lbs',
        ha='center', va='center', color=C495,
        fontsize=16, fontweight='bold', transform=ax.transAxes)
ax.axhline(0.04, color=C495, lw=3, xmin=0.08, xmax=0.92, alpha=0.6)

plt.savefig('reel_00_hook.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print('Slide 00 — Hook saved')

In [ ]:
# ── Slide 01 — 315 lbs REVEAL ────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)

# Title area (top 18%)
fig.text(0.5, 0.97, '3 PLATES', ha='center', va='top',
         color=C315, fontsize=46, fontweight='black')
fig.text(0.5, 0.90, '315 lbs', ha='center', va='top',
         color=WHITE, fontsize=28, fontweight='bold')
fig.text(0.5, 0.85, 'The first "real" squat milestone',
         ha='center', va='top', color=GRAY, fontsize=16)

# Chart (middle 55%)
ax = fig.add_axes([0.08, 0.30, 0.86, 0.52])
_style(ax)
reveal_chart_reel(315, C315, ax, '3 plates')
ax.set_title('')

# Bottom context
v315 = np.median(mc_results[315])
fig.text(0.5, 0.25, f'Out of 8.2 billion people...', ha='center', va='top',
         color=GRAY, fontsize=15)
fig.text(0.5, 0.20, f'only ≈{v315:.1f} million can squat this.',
         ha='center', va='top', color=WHITE, fontsize=17, fontweight='bold')
fig.text(0.5, 0.14, 'That includes every kid,\nevery grandparent, everyone.',
         ha='center', va='top', color=GRAY, fontsize=14)
fig.text(0.5, 0.06, '↓ 405 next ↓', ha='center', va='top',
         color=C405, fontsize=16, fontweight='bold')

plt.savefig('reel_01_315.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Slide 01 — 315 lbs: ≈{v315:.2f}M')

In [ ]:
# ── Slide 02 — 405 lbs REVEAL ────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)

fig.text(0.5, 0.97, '4 PLATES', ha='center', va='top',
         color=C405, fontsize=46, fontweight='black')
fig.text(0.5, 0.90, '405 lbs', ha='center', va='top',
         color=WHITE, fontsize=28, fontweight='bold')
fig.text(0.5, 0.85, 'Elite territory for most people',
         ha='center', va='top', color=GRAY, fontsize=16)

ax = fig.add_axes([0.08, 0.30, 0.86, 0.52])
_style(ax)
reveal_chart_reel(405, C405, ax, '4 plates')

v405 = np.median(mc_results[405])
fig.text(0.5, 0.25, 'The drop is brutal.', ha='center', va='top',
         color=GRAY, fontsize=15)
fig.text(0.5, 0.20, f'≈{v405:.2f}M people globally.',
         ha='center', va='top', color=WHITE, fontsize=17, fontweight='bold')
fig.text(0.5, 0.14, f'That\'s {np.median(mc_results[405])/np.median(mc_results[315])*100:.1f}% of the 315-lb club.',
         ha='center', va='top', color=C405, fontsize=14, fontweight='bold')
fig.text(0.5, 0.06, '↓ 495 changes everything ↓', ha='center', va='top',
         color=C495, fontsize=16, fontweight='bold')

plt.savefig('reel_02_405.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Slide 02 — 405 lbs: ≈{v405:.2f}M')

In [ ]:
# ── Slide 03 — 495 lbs REVEAL ────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)

fig.text(0.5, 0.97, '5 PLATES', ha='center', va='top',
         color=C495, fontsize=46, fontweight='black')
fig.text(0.5, 0.90, '495 lbs', ha='center', va='top',
         color=WHITE, fontsize=28, fontweight='bold')
fig.text(0.5, 0.85, 'The number most lifters never see',
         ha='center', va='top', color=GRAY, fontsize=16)

ax = fig.add_axes([0.08, 0.30, 0.86, 0.52])
_style(ax)
reveal_chart_reel(495, C495, ax, '5 plates')

v495 = np.median(mc_results[495])
pct495 = v495 / (WORLD_POP / 1e6) * 100
fig.text(0.5, 0.25, 'One in millions.', ha='center', va='top',
         color=GRAY, fontsize=15)
fig.text(0.5, 0.20, f'≈{v495:.2f}M people worldwide.',
         ha='center', va='top', color=WHITE, fontsize=17, fontweight='bold')
fig.text(0.5, 0.14, f'{pct495:.5f}% of all humans.\nThat\'s a sports stadium.',
         ha='center', va='top', color=C495, fontsize=14, fontweight='bold')
fig.text(0.5, 0.06, '↓ See the full drop-off ↓', ha='center', va='top',
         color=WHITE, fontsize=16, fontweight='bold')

plt.savefig('reel_03_495.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Slide 03 — 495 lbs: ≈{v495:.2f}M')

In [ ]:
# ── Slide 04 — Drop-off curve ─────────────────────────────────────────────────
X_TARGETS = np.linspace(100, 600, 200)
N_CURVE   = 10_000
rng3      = np.random.default_rng(7)

lift_c  = rng3.triangular(df_r['LiftLo'].values, df_r['LiftMode'].values,
                           df_r['LiftHi'].values, size=(N_CURVE, R))
msf_c   = rng3.uniform(msf_v * 0.85, msf_v * 1.15, size=(N_CURVE, R))
fsf_c   = rng3.uniform(fsf_v * 0.85, fsf_v * 1.15, size=(N_CURVE, R))
mmed_c  = rng3.uniform(df_r['MMedLo'].values, df_r['MMedHi'].values, size=(N_CURVE, R))
fmed_c  = rng3.uniform(df_r['FMedLo'].values, df_r['FMedHi'].values, size=(N_CURVE, R))
msig_c  = rng3.uniform(0.28, 0.38, size=(N_CURVE, 1, 1))
fsig_c  = rng3.uniform(0.29, 0.40, size=(N_CURVE, 1, 1))

mcnt_c  = (pop * lift_c * msf_c * FRAC_MALE)[:, :, None]
fcnt_c  = (pop * lift_c * fsf_c * (1 - FRAC_MALE))[:, :, None]
mmed_c3 = mmed_c[:, :, None]
fmed_c3 = fmed_c[:, :, None]
ln_x    = np.log(X_TARGETS)[None, None, :]

pm_c = stats.norm.sf((ln_x - np.log(mmed_c3)) / msig_c)
pf_c = stats.norm.sf((ln_x - np.log(fmed_c3)) / fsig_c)

curve_sims = (mcnt_c * pm_c + fcnt_c * pf_c).sum(axis=1) / 1e6
curve_med  = np.median(curve_sims, axis=0)
curve_lo   = np.percentile(curve_sims, 2.5,  axis=0)
curve_hi   = np.percentile(curve_sims, 97.5, axis=0)

# Portrait reel version
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, 'THE DROP-OFF', ha='center', va='top',
         color=WHITE, fontsize=36, fontweight='black')
fig.text(0.5, 0.91, '100 → 600 lbs, full picture', ha='center', va='top',
         color=GRAY, fontsize=16)

ax = fig.add_axes([0.10, 0.22, 0.86, 0.65])
_style(ax)
ax.fill_between(X_TARGETS, curve_lo, curve_hi, color=WHITE, alpha=0.10, label='95% CI')
ax.plot(X_TARGETS, curve_med, color=WHITE, lw=2.5, label='Median estimate')

for t, color, lbl in zip(MILESTONES, M_COLORS, M_LABELS):
    idx = int(np.argmin(np.abs(X_TARGETS - t)))
    ax.axvline(t, color=color, lw=2.0, ls='--', alpha=0.8)
    ax.scatter([t], [curve_med[idx]], color=color, s=100, zorder=5)
    ax.text(t + 5, curve_med[idx] * 1.5, f'{lbl}\n{curve_med[idx]:.2f}M',
            color=color, fontsize=9.5, va='bottom')

ax.set_yscale('log')
ax.set_xlabel('Squat (lbs)', fontsize=13)
ax.set_ylabel('People worldwide (M, log)', fontsize=12)
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3g}M'))

fig.text(0.5, 0.18, 'Every step up = exponential cut.',
         ha='center', va='top', color=WHITE, fontsize=15, fontweight='bold')
fig.text(0.5, 0.13, '315→405 cuts the club by ~80%.\n405→495 cuts it by ~85% again.',
         ha='center', va='top', color=GRAY, fontsize=13)

plt.savefig('reel_04_dropoff.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print('Slide 04 — Drop-off saved')

In [ ]:
# ── Slide 05 — Squat vs Bench comparison (context + retention hook) ──────────
# Bench numbers from bench_model for comparison
BENCH_ESTIMATES = {
    225: 9.27,   # median from bench_world_model
    315: 1.08,
    405: 0.12,
}
squat_meds = {t: np.median(mc_results[t]) for t in MILESTONES}

fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, 'SQUAT vs BENCH', ha='center', va='top',
         color=WHITE, fontsize=34, fontweight='black')
fig.text(0.5, 0.91, 'Same plate count, totally different rarity',
         ha='center', va='top', color=GRAY, fontsize=15)

ax = fig.add_axes([0.08, 0.35, 0.88, 0.50])
_style(ax)

# Paired bars: squat milestone vs equivalent bench milestone
pairs = [
    ('Bench 225\nvs Squat 315', BENCH_ESTIMATES[225], squat_meds[315], '2-plate bench', '3-plate squat'),
    ('Bench 315\nvs Squat 405', BENCH_ESTIMATES[315], squat_meds[405], '3-plate bench', '4-plate squat'),
    ('Bench 405\nvs Squat 495', BENCH_ESTIMATES[405], squat_meds[495], '4-plate bench', '5-plate squat'),
]

x = np.arange(len(pairs))
w = 0.35
for i, (label, bench_v, squat_v, blbl, slbl) in enumerate(pairs):
    b1 = ax.bar(i - w/2, bench_v, w, color='#888888', alpha=0.85, label='Bench' if i==0 else '')
    b2 = ax.bar(i + w/2, squat_v, w, color=M_COLORS[i], alpha=0.85, label='Squat' if i==0 else '')
    ax.text(i - w/2, bench_v + 0.05, f'{bench_v:.1f}M', ha='center', va='bottom',
            color=WHITE, fontsize=10, fontweight='bold')
    ax.text(i + w/2, squat_v + 0.05, f'{squat_v:.2f}M', ha='center', va='bottom',
            color=M_COLORS[i], fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(['225B / 315S', '315B / 405S', '405B / 495S'],
                    color=WHITE, fontsize=11)
ax.set_ylabel('People worldwide (M)', fontsize=12)
ax.legend(fontsize=11, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.set_title('Bench (gray) vs Squat (color) — same plate count', color=WHITE, fontsize=12)

fig.text(0.5, 0.30, 'The squat equivalents are\nalways rarer.', ha='center', va='top',
         color=WHITE, fontsize=16, fontweight='bold')
fig.text(0.5, 0.21, 'Fewer people squat than bench.\nHigher absolute weights. Steeper tail.',
         ha='center', va='top', color=GRAY, fontsize=13)
fig.text(0.5, 0.10, 'If you squat 4 plates,\nyou\'re in an extremely rare club.',
         ha='center', va='top', color=C405, fontsize=15, fontweight='bold')

plt.savefig('reel_05_comparison.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print('Slide 05 — Comparison saved')

In [ ]:
# ── Slide 06 — Regional breakdown for 315 ────────────────────────────────────
TARGET_RB = 315
per_r  = mc_per_region[TARGET_RB]
total  = per_r.sum(axis=1)

reg_med = np.median(per_r, axis=0)
reg_hi  = np.percentile(per_r, 97.5, axis=0)
tot_med = np.median(total)
tot_hi  = np.percentile(total, 97.5)

order   = np.argsort(reg_med)
all_med = np.append(reg_med[order], tot_med)
all_hi  = np.append(reg_hi[order],  tot_hi)
labels  = [df_r['Region'].iloc[i] for i in order] + ['TOTAL']
n       = len(labels)
bar_c   = [C315] * (n-1) + [C495]

fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
fig.text(0.5, 0.97, 'WHERE DO THEY LIVE?', ha='center', va='top',
         color=WHITE, fontsize=28, fontweight='black')
fig.text(0.5, 0.92, '315-lb squatters by region', ha='center', va='top',
         color=C315, fontsize=18, fontweight='bold')

ax = fig.add_axes([0.32, 0.08, 0.60, 0.80])
_style(ax)
y = np.arange(n)
ax.axhline(n - 1.5, color=GRID, lw=1.2, ls='--', alpha=0.7)

for i in range(n):
    ax.barh(i, all_hi[i],  color=bar_c[i], alpha=0.18, height=0.65, edgecolor='none')
    ax.barh(i, all_med[i], color=bar_c[i], alpha=0.88, height=0.65, edgecolor='none')
    ax.text(all_hi[i] * 1.04, i, f'{all_med[i]:.2f}M',
            va='center', color=WHITE, fontsize=9,
            fontweight='bold' if i == n-1 else 'normal')

ax.set_yticks(y)
ax.set_yticklabels(labels, color=WHITE, fontsize=9)
ax.set_xlabel('People (millions)', fontsize=11)
ax.invert_yaxis()
ax.set_title('', color=WHITE)

plt.savefig('reel_06_regional.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print('Slide 06 — Regional breakdown saved')

In [ ]:
# ── Slide 07 — Summary / CTA ─────────────────────────────────────────────────
fig = plt.figure(figsize=(REEL_W, REEL_H), facecolor=BG)
ax  = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor(BG)
ax.axis('off')

ax.axhline(0.93, color=C315, lw=4, xmin=0.08, xmax=0.92)

ax.text(0.5, 0.88, 'THE NUMBERS', ha='center', va='top',
        color=WHITE, fontsize=32, fontweight='black', transform=ax.transAxes)

for i, (t, c, lbl) in enumerate(zip(MILESTONES, M_COLORS, ['3 plates','4 plates','5 plates'])):
    med = np.median(mc_results[t])
    pct = med / (WORLD_POP/1e6) * 100
    y_base = 0.76 - i * 0.18
    ax.text(0.10, y_base, f'{t} lbs  ({lbl})', ha='left', va='top',
            color=c, fontsize=19, fontweight='bold', transform=ax.transAxes)
    ax.text(0.10, y_base - 0.055, f'≈ {med:.2f}M people  ({pct:.4f}% of Earth)',
            ha='left', va='top', color=WHITE, fontsize=14, transform=ax.transAxes)
    ax.axhline(y_base - 0.085, color=GRID, lw=0.8, xmin=0.08, xmax=0.92, alpha=0.5,
               transform=ax.transAxes)

ax.text(0.5, 0.25, 'Model: 3.9M OPL lifts + 10-region world\n100,000 Monte Carlo trials',
        ha='center', va='top', color=GRAY, fontsize=12, transform=ax.transAxes)

ax.text(0.5, 0.14, 'How much do you squat?\nDrop it below 👇',
        ha='center', va='top', color=WHITE, fontsize=18, fontweight='bold',
        transform=ax.transAxes)

ax.axhline(0.05, color=C495, lw=3, xmin=0.08, xmax=0.92, alpha=0.7,
           transform=ax.transAxes)

plt.savefig('reel_07_summary.png', dpi=REEL_DPI, bbox_inches='tight', facecolor=BG)
plt.show()
print('Slide 07 — Summary/CTA saved')

---
## 📊 Standard Analysis Charts
*Landscape versions for writeup / thumbnail use.*

In [ ]:
# ── Landscape: all three milestones side by side ─────────────────────────────
def reveal_card(target, color, ax, show_ylabel=True):
    sims     = mc_results[target]
    med      = np.median(sims)
    lo, hi   = np.percentile(sims, [2.5, 97.5])
    pct      = med / (WORLD_POP / 1e6) * 100
    x = np.linspace(20, 750, 1200)
    pdf_m = _lognorm.pdf(x, _m_sig, scale=_global_m_med)
    pdf_f = _lognorm.pdf(x, _f_sig, scale=_global_f_med)
    ax.plot(x, pdf_m, color=MALE, lw=2.0, alpha=0.85, label='Male')
    ax.plot(x, pdf_f, color=FEM,  lw=2.0, alpha=0.85, label='Female')
    mask = x >= target
    ax.fill_between(x[mask], pdf_m[mask], color=MALE, alpha=0.30)
    ax.fill_between(x[mask], pdf_f[mask], color=FEM,  alpha=0.30)
    ax.axvline(target, color=color, lw=2.2, ls='--', alpha=0.9)
    y_top = max(pdf_m.max(), pdf_f.max())
    ax.text(target + 6, y_top * 0.94, f'{target} lbs', color=color, fontsize=12, fontweight='bold')
    ax.text(0.97, 0.95,
            f'≈ {med:.2f}M worldwide\n95% CI: {lo:.2f}M – {hi:.2f}M\n≈ {pct:.4f}% of humans',
            transform=ax.transAxes, ha='right', va='top', color=WHITE, fontsize=10,
            bbox=dict(facecolor=GRID, edgecolor='none', pad=6))
    ax.set_xlim(20, 750)
    ax.set_xlabel('Squat (lbs)', fontsize=10)
    if show_ylabel: ax.set_ylabel('Density', fontsize=10)
    ax.legend(fontsize=9, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID, loc='upper left')

fig, axes = make_fig(15, 6, cols=3)
fig.suptitle('All Three Squat Milestones — Median + 95% CI',
             color=WHITE, fontsize=14, fontweight='bold', y=1.01)
for ax, m, color, tag in zip(axes, MILESTONES, M_COLORS, M_TAGS):
    reveal_card(m, color, ax, show_ylabel=(ax == axes[0]))
    ax.set_title(tag, color=color, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('10_all_three.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# ── Landscape: full drop-off curve ───────────────────────────────────────────
fig, ax = make_fig(13, 6)
ax.fill_between(X_TARGETS, curve_lo, curve_hi, color=WHITE, alpha=0.12, label='95% CI')
ax.plot(X_TARGETS, curve_med, color=WHITE, lw=2.2, label='Median estimate')

for t, color, lbl in zip(MILESTONES, M_COLORS, M_LABELS):
    idx = int(np.argmin(np.abs(X_TARGETS - t)))
    ax.axvline(t, color=color, lw=1.5, ls='--', alpha=0.7)
    ax.scatter([t], [curve_med[idx]], color=color, s=80, zorder=5)
    ax.text(t + 4, curve_med[idx] * 1.4, f'{lbl}\n{curve_med[idx]:.2f}M',
            color=color, fontsize=9.5, va='bottom')

ax.set_yscale('log')
ax.set_xlabel('Squat (lbs)', fontsize=12)
ax.set_ylabel('People worldwide (millions, log scale)', fontsize=12)
ax.set_title('Global Squat Capability — Full Drop-Off Curve',
             color=WHITE, fontsize=14, fontweight='bold')
ax.legend(fontsize=10, facecolor=PANEL, labelcolor=WHITE, edgecolor=GRID)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.3g}M'))
plt.tight_layout()
plt.savefig('08_dropoff.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## ✅ Final Numbers

In [ ]:
print('='*60)
print('SQUAT WORLD MODEL — FINAL ESTIMATES')
print('='*60)
for t in MILESTONES:
    v   = mc_results[t]
    med = np.median(v)
    lo  = np.percentile(v, 2.5)
    hi  = np.percentile(v, 97.5)
    pct = med / (WORLD_POP/1e6) * 100
    in_x = WORLD_POP / (med * 1e6)
    print(f'\n{t} lbs ({t//45} plates):')
    print(f'  Median: {med:.2f}M  [{lo:.2f}M – {hi:.2f}M]')
    print(f'  {pct:.5f}% of all humans')
    print(f'  Roughly 1 in {in_x:,.0f} people on Earth')

---
## 📎 Appendix — Assumptions

### What counts as 'can squat X'
This model estimates the number of people who **currently** have the strength capacity for a free-weight back squat at that weight. It does **not** include:
- People who could reach it with training but haven't
- People using Smith machine, leg press, or other non-barbell variations
- Equipped squat (suit adds 100–200+ lbs)

### Who is excluded from the denominator
We use the **full 8.2 billion world population** as the denominator for % calculations.  
The **16-70 active population** (4.674B) is the pool from which squatters are drawn.  
This means children, elderly (70+), and people in regions with near-zero gym access are correctly counted as *unable to hit these milestones* rather than excluded.

### Squat vs bench fractions
The key insight driving lower squat numbers vs. bench:
1. **Absolute numbers are higher** — 315 squat requires more total strength than 225 bench
2. **Fewer people barbell squat** — squat rack intimidation, mobility requirements, and 'leg day' skipping means a smaller fraction of gym-goers back squat vs. those who bench press
3. **Women's squat fraction is higher relative to bench** — squat rack culture for glute training has grown, but total participation is still lower than male squat rates

### Caveats
- Developing-region lift rates are weakest inputs
- 495 lbs: deep tail, treat as order-of-magnitude
- Squat form standards vary (parallel vs. ATG vs. high-bar vs. low-bar)
- Model reflects current capability, not potential